In [1]:
from pdfminer.high_level import extract_text
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup
import re
import spacy, json

#### PDF OCR

In [2]:
text = extract_text("data/Draft 12_15_22 - Project Hail Mary Screenplay.pdf")
with open("phm_script_raw_draft.txt", "w", encoding="utf-8") as f:
    f.write(text)

#### EPUB to txt

In [12]:
BLOCK_TAGS = {"p", "div", "br", "h1", "h2", "h3", "h4", "h5", "h6", "li", "tr"}

def epub_to_text(epub_path):
    book = epub.read_epub(epub_path)
    chapters = []

    for item in book.get_items():
        if item.get_type() == ebooklib.ITEM_DOCUMENT:
            soup = BeautifulSoup(item.get_content(), "html.parser")

            # Insert newline markers at block boundaries only
            for tag in soup.find_all(BLOCK_TAGS):
                tag.insert_before("\n")
                tag.insert_after("\n")

            # Now get_text with no separator — inline tags just yield their text naturally
            text = soup.get_text(separator="")

            # Clean up
            text = re.sub(r" {2,}", " ", text)        # collapse extra spaces
            text = re.sub(r"\n{3,}", "\n\n", text)    # collapse extra blank lines
            text = re.sub(r" \n", "\n", text)         # trailing spaces before newlines
            text = re.sub(r"\n ", "\n", text)         # leading spaces after newlines

            chapters.append(text.strip())

    return "\n\n".join(chapters)

full_text = epub_to_text("data/Project Hail Mary.epub")
with open("book_raw.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

In [16]:
import re
import json

OPEN_QUOTE  = "\u201c"
CLOSE_QUOTE = "\u201d"
ATTRIBUTION = re.compile(
    r',?\s*(he says|Rocky says|he asks|Rocky asks|Rocky calls|he adds'
    r'|he says again|Rocky says out of nowhere|he asks again)[^"]*',
    re.IGNORECASE,
)
MUSICAL    = re.compile(r'^[♩♪♫♬]+$')
FORCE_CAPS = {
    "Rocky", "Grace", "Hail", "Mary", "EVA", "Astrophage", "Eridian",
    "Erid", "Adrian", "Petrova", "Earth", "Tau", "Ceti", "Stryan",
}

def extract_speech(line: str) -> str:
    fragments = re.findall(
        rf'{OPEN_QUOTE}([^{CLOSE_QUOTE}]+){CLOSE_QUOTE}|"([^"]+)"', line
    )
    if fragments:
        parts = [a or b for a, b in fragments]
        parts = [p.rstrip(", ") for p in parts]
        # Join fragments with period+space only if the previous fragment
        # doesn't already end with sentence-ending punctuation
        result = parts[0]
        for part in parts[1:]:
            if result and result[-1] in '.!?…':
                result += ' ' + part
            else:
                result += '. ' + part
        return result.strip()
    cleaned = ATTRIBUTION.sub(". ", line)
    cleaned = re.sub(r'\.(\s*\.)+', '.', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

def split_sentences(line):
    # Split on . ! ? … and also on : when followed by a capital letter
    return re.split(r'(?<=[.!?…])\s+|(?<=:)\s+(?=[A-Z])', line)

def tokenize(text):
    return re.findall(r'\d+\.?\d*|[^\s.,!?;:\"\'""\'\'…—\-]+', text)

# Load and clean lines from both sources
with open("rocky_lines/rocky_book_lines.txt", encoding="utf-8") as f:
    raw_lines = [l.rstrip("\n") for l in f if l.strip()]
cleaned_lines = [extract_speech(l) for l in raw_lines]

with open("rocky_lines/rocky_movie_lines.txt", encoding="utf-8") as f:
    raw_script_lines = [l.rstrip("\n") for l in f if l.strip()]
cleaned_lines.extend([extract_speech(l) for l in raw_script_lines])

# First pass — find all genuinely mid-sentence capitalised tokens
mid_sentence_caps = set()
for line in cleaned_lines:
    for sentence in split_sentences(line):
        tokens = tokenize(sentence)
        for i, t in enumerate(tokens):
            if t and t[0].isupper() and i > 0:
                mid_sentence_caps.add(t)

# Second pass — build vocab with casing rules
vocab = set()
for line in cleaned_lines:
    for sentence in split_sentences(line):
        tokens = tokenize(sentence)
        for i, t in enumerate(tokens):
            if not t or MUSICAL.match(t):
                continue
            if t in FORCE_CAPS:
                # Always keep capitalised
                vocab.add(t)
                # Also add lowercase if it has a non-proper meaning
                vocab.add(t.lower())
            elif t in mid_sentence_caps:
                # Genuine mid-sentence proper noun — keep capitalised
                vocab.add(t)
            elif i == 0 and t[0].isupper():
                # Sentence-start cap — add only lowercase
                vocab.add(t.lower())
            else:
                vocab.add(t)

with open("rocky_vocab.json", "w", encoding="utf-8") as f:
    json.dump(sorted(vocab), f, indent=2, ensure_ascii=False)

with open("rocky_lines_clean.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(cleaned_lines))

print(f"Total unique tokens: {len(vocab)}")

caps = sorted(w for w in vocab if w and w[0].isupper())
print(f"\nCapitalised tokens kept ({len(caps)}) — review these:")
for w in caps:
    print(f"  {w}")

numbers = sorted(w for w in vocab if any(c.isdigit() for c in w))
print(f"\nNumeric tokens ({len(numbers)}):")
for n in numbers:
    print(f"  {n}")

Total unique tokens: 1207

Capitalised tokens kept (48) — review these:
  A
  Adrian
  Astronomy
  Astrophage
  Blip
  Celsius
  Ceti
  Circle
  EATS
  EVA
  Earth
  Eighteen
  Erid
  Eridani
  Eridian
  Eridians
  Five
  Four
  Friend
  Grace
  Hail
  I
  IR
  Mark
  Mary
  Medium
  Morning
  Nine
  Paul
  Petrova
  Petrovascope
  Planet
  Ringo
  Rocky
  Rough
  Save
  Six
  Sol
  Tanks
  Tau
  Taumoeba
  Ten
  Texture
  Threeworld
  Twelve
  Twenty
  Understand
  Venus

Numeric tokens (17):
  0.61
  1.1
  100
  12
  15
  186.3
  198.8
  20.48
  265
  35
  4.5
  42
  6.64
  80.
  82.5
  86.
  928
